# Mean-Field Simulation of a Quantum Dot Coupled to a Bosonic Bath

This notebook implements a numerical simulation of Andreev bound states
and the Josephson current in a superconducting quantum dot including

- spin–orbit interaction
- Zeeman splitting
- Coulomb interaction (mean-field)
- coupling to a bosonic environment.

The bosonic bath induces transitions between fermionic states,
which are treated using a Lindblad master equation.

The workflow of the simulation is:

1. Define model parameters
2. Construct dot eigenstates
3. Compute hybridization matrices
4. Calculate Andreev bound states
5. Solve the self-consistent mean-field equations
6. Compute steady-state occupations
7. Evaluate the Josephson current

*Note:* This notebook demonstrates the numerical implementation of the
method. Results have not been benchmarked against reference calculations
and are shown for illustration purposes.

## Imports

In [ ]:
import cmath
import math
import numpy as np
import matplotlib.pyplot as plt
from numpy import linalg as LA

from scipy.integrate import solve_ivp
from scipy import integrate
from scipy.linalg import null_space

from scipy.optimize import fsolve
from scipy.integrate import solve_bvp
from scipy.integrate import quad_vec

from itertools import chain

from concurrent.futures import ProcessPoolExecutor

In [ ]:
%matplotlib notebook

## Model parameters

### Parameters of the dot

In [ ]:
# length of the dot
L = 1.7

# orbital number
l = 2

# coupling srength to the left/right lead
t_L = 1
t_R = t_L

# effective mass
mx = 9

# SC gap
Delta = 1

# Zeeman field
b_x = 0.21
b_z = 0.2

# SOI coupling
gamma_SO = 0.14

# Coulomb, backgate voltage
Ec = 0.1
Vg_min_mu = -0.4

### Parameters of the reservoirs

In [ ]:
# Andreev-continuum coupling rate
lambda_coupl = 0.1 

# linewidth of the resonant mode
damping_param = 0.1 

# resonant microwave frequency
Omega = 0.01

# continuum temperature
T_ferm = 0.5

# bosonic bath temperature
T_bos = 0.2

# coupling to the Ohmic environment
alpha_0 = 0.001 

# high-frequency cutoff of the Ohmic bath
omega_c = 1

### Phase grid

In [ ]:
phi_0 = np.linspace(0, 2*np.pi, num=31)

phi_0[0] = 0.001
phi_0[-1] -= 0.001

### Pauli matrices

In [ ]:
tau_z = np.block([[np.eye(4), np.zeros((4, 4))], 
                  [np.zeros((4, 4)), -np.eye(4)]])

### Initialization of Mean-Field Parameters

In [ ]:
MF_param = np.zeros((len(phi_0), 2*l))
accur_par = 6*10**(-3)
abs_diff = np.full((len(phi_0), 2*l), np.inf)

## Distribution functions

In [ ]:
def bose_distrib(x, T):
    """This function implements Bose distribution for the microwave environment:
       x - energy of the state,
       T - temperature of the bath."""
    
    if x>=0:
        n_B = 1/(np.exp(x/T)-1)
        return n_B
    else:
        n_B = 1/(np.exp(-x/T)-1)
        return -1-n_B

In [ ]:
def fermi_distrib(x, T):
    """This function implements Fermi distribution for the fermionic continuum:
       x - energy of the state,
       T - temperature of the fermionic bath."""
    
    return 1/(np.exp(x/T)+1)

In [ ]:
def spectral_density(x, lambda_coupl, alpha_0, damping_param, Omega, omega_c):
    """This function implements the spectral density - Lorenzian and Ohmic - for the microwave environment:
       x - energy of the state,
       lambda_coupl, alpha_0, damping_param, Omega, omega_c - bath parameters."""
    
    J = lambda_coupl**2 * damping_param * (1/((x - Omega)**2 + damping_param**2/4) - \
                                                   1/((x + Omega)**2 + damping_param**2/4))
    
    J_ohm = alpha_0 * x * np.exp(-np.abs(x)/omega_c)
    
    return J + J_ohm

## Finding free dot eigenenergies and eigenfunctions -  hopping parameters

The eigenfunctions of the isolated dot are calculated including
spin–orbit coupling.<br> Lead-dot hopping parameters are constructed from them.

### Construction of the free dot Hamiltonian

In [ ]:
def chi_n_sigma(x, n, sigma):
    """
    Eigenfunction of the isolated dot including spin–orbit coupling.
    """
    
    q = mx*gamma_SO
    k_n = n*np.pi/L
    
    cos_n = (k_n + sigma*q)/np.sqrt(2*(k_n**2 + q**2))
    sin_n = (k_n - sigma*q)/np.sqrt(2*(k_n**2 + q**2))
    
    f = np.exp(-1j*sigma*q*(x - L/2)) * (cos_n * np.exp(1j*k_n*(x-L/2)) + \
                                   sin_n * np.exp(-1j*k_n*(x-L/2)))
    
    return f/np.sqrt(L)

In [ ]:
# Example wavefunction plot

x_plot = np.linspace(-L/2, L/2, 100)

plt.xlabel("x")
plt.ylabel("Wavefunction")

plt.plot(x_plot, np.real(chi_n_sigma(x_plot, 3, -1)))
plt.plot(x_plot, np.imag(chi_n_sigma(x_plot, 3, -1)))

plt.grid()

In [ ]:
def cos_n_s(n, sigma):
     """
    Plane-wave mixing coefficient in the SO-coupled dot eigenstate.
    """ 
        
    k_n = n*np.pi/L
    q = mx*gamma_SO
    
    cos_n = (k_n + sigma*q)/np.sqrt(2*(k_n**2 + q**2))
    
    return cos_n   

In [ ]:
def J_mn_sigma(m, n, sigma):
    """
    Spin-flip coupling between orbitals m and n induced by
    spin–orbit interaction and transverse Zeeman field.
    """
    
    q = mx*gamma_SO
    k_n = n*np.pi/L
    k_m = m*np.pi/L
    
    f_s_mn = cos_n_s(n, sigma) * cos_n_s(m, -sigma) / (k_n - k_m - 2*q*sigma) -\
             cos_n_s(n, -sigma) * cos_n_s(m, sigma) / (k_n - k_m + 2*q*sigma) +\
             cos_n_s(n, sigma) * cos_n_s(m, sigma) / (k_n + k_m - 2*q*sigma) -\
             cos_n_s(n, -sigma) * cos_n_s(m, -sigma) / (k_n + k_m + 2*q*sigma)
    
    if (-1)**(n+m) == 1:
        J = 2*b_x * f_s_mn / L * (-sigma)*np.sin(q*L)
    if (-1)**(n+m) == -1:
        J = 2*b_x * f_s_mn / L * (-1j)*np.cos(q*L)
    
    return J

In [ ]:
# number of dot levels used in the truncated basis
N = 20

# orbital indices
n_arr = np.arange(1, N + 1)

# spin–orbit momentum shift
q = mx * gamma_SO

# quantized momenta of the dot
k_n = n_arr * np.pi / L

# single-particle energies of the isolated dot
eps_n = k_n**2 / (2 * mx) - mx * gamma_SO**2 / 2 + Vg_min_mu


# spin-flip coupling matrices between orbitals
J_min = np.zeros((N, N), dtype='complex')
J_pl = np.zeros_like(J_min)

# compute coupling elements J_{mn}
for n in range(N):
    for m in range(N):
        J_min[n, m] = J_mn_sigma(n+1, m+1, -1)
        J_pl[m, n] = np.conjugate(J_min[n, m])

In [ ]:
# effective Hamiltonian
H_2 = np.zeros((2*N, 2*N), dtype='complex')

diagonal_elements = np.array([(eps + b_z, eps - b_z) for eps in eps_n]).flatten()
H_diag = np.diag(diagonal_elements)

for n in range(N):
    for m in range(N):
        H_2[2*n, 2*m+1] = J_min[n, m]
        H_2[2*n+1, 2*m] = J_pl[n, m]
        
H_2 += H_diag

# finding the eigenenergies and eigenfunctions
E_2, Psi_2 = LA.eigh(H_2)

for i in range(2*N):
    sum_i = np.real(np.sum(Psi_2[:, i]**2))
    Psi_2[:, i] /= sum_i

# hopping parameters    
even_rows = Psi_2[::2, :]
u = np.sum(xi_n[:, np.newaxis] * even_rows, axis=0)
t_L_up = np.exp(1j*L*q) * np.sum(m_one_arr[:, np.newaxis] * xi_n[:, np.newaxis] * even_rows, axis=0)


odd_rows = Psi_2[1::2, :]
v = np.sum(xi_n[:, np.newaxis] * odd_rows, axis=0)
t_L_down = np.exp(-1j*L*q) * np.sum(m_one_arr[:, np.newaxis] * xi_n[:, np.newaxis] * odd_rows, axis=0)

In [ ]:
# left- and right-leads hybridization matrices

Gamma_L = np.zeros((2*l, 2*l), dtype='complex')
Gamma_R = np.zeros_like(Gamma_L)
F_R = np.zeros_like(Gamma_L)
F_L = np.zeros_like(Gamma_L)

for mu in range(4):
    for nu in range(4):
        Gamma_R[mu, nu] = np.conjugate(u[mu])*u[nu] + np.conjugate(v[mu])*v[nu]
        Gamma_L[mu, nu] = u[mu]*np.conjugate(u[nu]) + v[mu]*np.conjugate(v[nu])
    
        F_R[mu, nu] = u[mu]*v[nu] - u[nu]*v[mu]
        F_L[mu, nu] = np.conjugate(u[mu]*v[nu] - u[nu]*v[mu])  

Gamma_L *= t_L**2
Gamma_R *= t_L**2
F_L *= t_L**2
F_R *= t_L**2

## Mean-Field Hamiltonian and ABSs

In [ ]:
def G_inv(x, i, MF_param):
    """
    Computes det(G^{-1}(x)) for the interacting quantum dot.

    The inverse Green function includes the dot Hamiltonian,
    mean-field interaction terms, and the hybridization terms
    from the left and right leads.

    The zeros of this determinant correspond to the
    Andreev bound state energies.
    
    Returns
    -------
    complex
    Determinant of the inverse Green's function det(G^{-1}(x)).
    """
    
    s_L = 1
    s_R = -1
    zero_m = np.zeros_like(Gamma_L)
    
    
    Lambda = -1 / cmath.sqrt(Delta**2 - x**2) \
            * (x * np.block([[(Gamma_L + Gamma_R), zero_m], [zero_m, np.conjugate(Gamma_L + Gamma_R)]]) \
            - Delta * np.block([[zero_m, np.exp(-1j * s_L * phi_0[i]/2) * np.conjugate(np.transpose(F_L))], \
                                [np.exp(1j * s_L * phi_0[i]/2) * F_L, zero_m]]) \
            - Delta * np.block([[zero_m, np.exp(-1j * s_R * phi_0[i]/2) * np.conjugate(np.transpose(F_R))], \
                                [np.exp(1j * s_R * phi_0[i]/2) * F_R, zero_m]]))
    
    G_inv = x * np.eye(4*l) - np.block([[np.diag(E_2[0:2*l]) + 2*Ec*np.diag(MF_param[i]), zero_m],\
                                        [zero_m, -np.diag(E_2[0:2*l]) - 2*Ec*np.diag(MF_param[i])]]) - Lambda
    
    return LA.det(G_inv)   

In [ ]:
# Computing ABSs

roots = np.zeros((len(phi_0), 4*l))

for i in range(len(phi_0)):
    roots[i, 0] = fsolve(G_inv, 0.6, args=(i, MF_param))  
    roots[i, 1] = fsolve(G_inv, 0.7, args=(i, MF_param))     
    roots[i, 2] = fsolve(G_inv, 0.8, args=(i, MF_param))  
    roots[i, 3] = fsolve(G_inv, 0.99, args=(i, MF_param))
    
    roots[i, 4] = fsolve(G_inv, -0.6, args=(i, MF_param))  
    roots[i, 5] = fsolve(G_inv, -0.7, args=(i, MF_param))     
    roots[i, 6] = fsolve(G_inv, -0.8, args=(i, MF_param))  
    roots[i, 7] = fsolve(G_inv, -0.99, args=(i, MF_param))   

In [ ]:
# plotting ABSs

plt.plot(phi_0/np.pi, roots[:,0], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,1], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,2], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,3], linewidth=1.5)


plt.title(r"$t_L={}, t_R={}, \gamma_{{SO}} = {},$"\
          .format(t_L, t_R, gamma_SO)+'\n'+
          r"$b_x={}, b_z = {}, L={}$"\
          .format(np.round(b_x, 2), np.round(b_z, 2), L), fontsize=13)



plt.grid()
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)
plt.ylabel(r'$E_A$, $\Delta$', fontsize=12)


## Transition rates between fermionic states:

## ABS to ABS transition rates

In [ ]:
def diagonal(x, i, MF_param):
    """
    Diagonalizes the effective dot Hamiltonian of the interacting quantum dot for phase index i.

    The Hamiltonian includes the dot energy levels, mean-field interaction
    terms, and the superconducting self-energy from the leads.

    Returns
    -------
    eigval : ndarray
        Eigenvalues of the Hamiltonian (quasiparticle energies).

    eigvec : ndarray
        Corresponding eigenvectors.
    """
    
    s_L = 1
    s_R = -1
    zero_m = np.zeros_like(Gamma_L)
    
    Lambda = -1 / np.sqrt(Delta**2 - x[i]**2) \
        * (x[i] * np.block([[(Gamma_L + Gamma_R), zero_m], [zero_m, (Gamma_L + Gamma_R)]]) \
        - Delta * np.block([[zero_m, np.exp(-1j * s_L * phi_0[i]/2) * np.conjugate(np.transpose(F_L))], \
                            [np.exp(1j * s_L * phi_0[i]/2) * F_L, zero_m]]) \
        - Delta * np.block([[zero_m, np.exp(-1j * s_R * phi_0[i]/2) * np.conjugate(np.transpose(F_R))], \
                            [np.exp(1j * s_R * phi_0[i]/2) * F_R, zero_m]]))


    H = np.block([[np.diag(E_2[0:2*l]) + 2*Ec*np.diag(MF_param[i]), zero_m],\
                  [zero_m, -np.diag(E_2[0:2*l]) - 2*Ec*np.diag(MF_param[i])]]) + Lambda
         
    eigval, eigvec = LA.eigh(H)
    
    return eigval, eigvec 

In [ ]:
eigv_0 = np.zeros((len(phi_0), 4*l))
eigvec_0 = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

eigv_1 = np.zeros((len(phi_0), 4*l))
eigvec_1 = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

eigv_2 = np.zeros((len(phi_0), 4*l))
eigvec_2 = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

eigv_3 = np.zeros((len(phi_0), 4*l))
eigvec_3 = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

eigv_4 = np.zeros((len(phi_0), 4*l))
eigvec_4 = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

eigv_5 = np.zeros((len(phi_0), 4*l))
eigvec_5 = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

eigv_6 = np.zeros((len(phi_0), 4*l))
eigvec_6 = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

eigv_7 = np.zeros((len(phi_0), 4*l))
eigvec_7 = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

for i in range(len(phi_0)):
    
    eigv_0[i], eigvec_0[i] = diagonal(roots[:, 0], i, MF_param)
    eigv_1[i], eigvec_1[i] = diagonal(roots[:, 1], i, MF_param)
    eigv_2[i], eigvec_2[i] = diagonal(roots[:, 2], i, MF_param)
    eigv_3[i], eigvec_3[i] = diagonal(roots[:, 3], i, MF_param)

    eigv_4[i], eigvec_4[i] = diagonal(roots[:, 4], i, MF_param)
    eigv_5[i], eigvec_5[i] = diagonal(roots[:, 5], i, MF_param)
    eigv_6[i], eigvec_6[i] = diagonal(roots[:, 6], i, MF_param)
    eigv_7[i], eigvec_7[i] = diagonal(roots[:, 7], i, MF_param)

In [ ]:
# Eigenvectors corresponding to the Andreev energies

eta_0 = eigvec_0[:, :, 4]
eta_1 = eigvec_1[:, :, 5]
eta_2 = eigvec_2[:, :, 6]
eta_3 = eigvec_3[:, :, 7]

eta_4 = eigvec_4[:, :, 3]
eta_5 = eigvec_5[:, :, 2]
eta_6 = eigvec_6[:, :, 1]
eta_7 = eigvec_7[:, :, 0]

eta_arr = np.stack([eta_0, eta_1, eta_2, eta_3, eta_4, eta_5, eta_6, eta_7], axis=0)

### Bogoliubov transformation

In [ ]:
## Constructing the Bogoliubov transformation between dot fermions and Andreev quasiparticles
## and extracting particle–hole weights and orbital contributions of the Andreev bound states 

M = np.zeros((len(phi_0), 8, 8), dtype='complex')

# Fill M[:, 0:4, :] with the conjugates of eta_0 to eta_3
M[:, 0, :] = np.conj(eta_0)
M[:, 1, :] = np.conj(eta_1)
M[:, 2, :] = np.conj(eta_2)
M[:, 3, :] = np.conj(eta_3)

# For rows 4 to 7, split and assign parts
for i, eta in enumerate([eta_0, eta_1, eta_2, eta_3]):
    M[:, 4 + i, 0:4] = eta[:, 4:]
    M[:, 4 + i, 4:] = eta[:, 0:4]
    
M_inv = LA.inv(M)
a = np.sum(np.abs(M_inv[:, 0:4, 4:])**2, axis=1)

# weight of each number of part operator n_i, i in [1, 4]
a_i = np.abs(M_inv[:, 0:4, 0:4])**2 - np.abs(M_inv[:, 4:, 0:4])**2

In [ ]:
def curr_matr(x, i):
    """
    Constructs the current operator matrix in Nambu space for a given
    energy x and superconducting phase index i.

    The matrix is derived from the phase-dependent self-energy
    contributions of the left and right superconducting leads.
    It enters the calculation of the Josephson current through
    the Andreev bound states.

    Returns
    -------
    I  :  ndarray
        Current operator matrix evaluated at energy x.
    """
    
    s_L = 1
    s_R = -1
    zero_m = np.zeros_like(Gamma_L)
    
    Lambda_L = x * np.block([[Gamma_L , zero_m], [zero_m, Gamma_L]]) \
             - Delta * np.block([[zero_m, np.exp(-1j * s_L * phi_0[i]/2) * np.conjugate(np.transpose(F_L))], \
                                [np.exp(1j * s_L * phi_0[i]/2) * F_L, zero_m]]) 
    
    Lambda_R = x * np.block([[Gamma_R , zero_m], [zero_m, Gamma_R]]) \
             - Delta * np.block([[zero_m, np.exp(-1j * s_R * phi_0[i]/2) * np.conjugate(np.transpose(F_R))], \
                                [np.exp(1j * s_R * phi_0[i]/2) * F_R, zero_m]])
        
    I = -1 / np.sqrt(Delta**2 - x**2) / (2*1j) * (s_L * Lambda_L + s_R * Lambda_R)
    
    return I

In [ ]:
def curr_nl(E_n, E_lamb, eta_n, eta_lamb, i):
    """
    Computes the current matrix element between two Andreev bound states.

    The matrix element is evaluated between eigenvectors η_n and η_λ
    corresponding to energies E_n and E_λ. It involves the current
    operator in Nambu space and the τ_z matrix, which accounts for the
    particle–hole structure of the Bogoliubov quasiparticles.

    Returns
    -------
    complex
        Current matrix element between the two Andreev states.
    """
            
    matr = tau_z @ curr_matr(E_lamb, i) - curr_matr(E_n, i) @ tau_z
    
    return np.conj(eta_n) @ matr @ eta_lamb    

In [ ]:
## Computing current matrix elements between the ABSs and corresponding transition rates

# Array of current matrix elements between all 4l ABSs
I_nl = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')

# Array of transition rates between all 4l ABSs
G_nl = np.zeros((len(phi_0), 4*l, 4*l))

for i in range(len(phi_0)):
    for n in range(4*l):
        for lamd in range(4*l):
            I_nl[i, n ,lamd] = curr_nl(roots[i, n], roots[i, lamd], eta_arr[n, i], eta_arr[lamd, i], i)

In [ ]:
## Calculating transition rates between the ABSs

for i in range(len(phi_0)):
    for n in range(4*l):
        for lamd in range(4*l):
            
            energy_diff = roots[i, n] - roots[i, lamd]
            J_spect = spectral_density(energy_diff, lambda_coupl, alpha_0, damping_param, Omega, omega_c)
            bose = bose_distrib(energy_diff, T_bos)
            
            G_nl[i, lamd, n] = 2*np.pi * np.abs(I_nl[i, n, lamd])**2 * J_spect * bose 
            
            if math.isnan(G_nl[i, lamd, n]):
                G_nl[i, lamd, n] = 0

In [ ]:
## Defining transition rates between the positive- and negative-energy ABSs

# Trans. rates between positive-energy ABSs
G_AL_pp = G_nl[:, 0:4, 0:4]
# Trans. rates between positive-to-negative ABSs
G_AL_pm = G_nl[:, 0:4, 4:8]
# Trans. rates between negative-to-positive ABSs
G_AL_mp = G_nl[:, 4:8, 0:4]

## Transition rates between ABSs and continuum (represented as density-of-states)

In [ ]:
def spectr(x, i, MF_param):
    """
    Computes the density-of-states matrix of the interacting
    quantum dot for energy x and phase index i.

    The spectral matrix is obtained from the retarded and advanced Green
    functions as A(x) = i (G_R - G_A). It includes the dot Hamiltonian,
    mean-field interaction terms, and the hybridization with the
    superconducting leads.

    The function diagonalizes the spectral matrix to obtain its
    eigenvalues and eigenvectors, which describe the spectral weights
    and corresponding quasiparticle states.

    Returns
    -------
    eigval_arr : ndarray
        Eigenvalues of the spectral matrix (spectral weights).

    eigvec_arr : ndarray
        Corresponding eigenvectors of the spectral matrix.
    """

    eigval_arr = np.zeros((8))
    eigvec_arr = np.zeros((8, 8), dtype='complex')
    
    s_L = 1
    s_R = -1
    zero_m = np.zeros_like(Gamma_L)
    
    Lambda = 1 / np.sqrt(x**2 - Delta**2) \
        * (x * np.block([[(Gamma_L + Gamma_R), zero_m], [zero_m, (Gamma_L + Gamma_R)]]) \
        - Delta * np.block([[zero_m, np.exp(-1j * s_L * phi_0[i]/2) * np.conjugate(np.transpose(F_L))], \
                                [np.exp(1j * s_L * phi_0[i]/2) * F_L, zero_m]]) \
        - Delta * np.block([[zero_m, np.exp(-1j * s_R * phi_0[i]/2) * np.conjugate(np.transpose(F_R))], \
                                [np.exp(1j * s_R * phi_0[i]/2) * F_R, zero_m]]))

    G_R = LA.inv(x * np.eye(8) - np.block([[np.diag(E_2[0:4]) + 2*Ec*np.diag(MF_param[i]), zero_m],\
                                           [zero_m, -np.diag(E_2[0:4]) - 2*Ec*np.diag(MF_param[i])]]) \
          + 1j * np.sign(x) * Lambda)

    G_A = np.transpose(np.conj(G_R))

    Spectr = 1j * (G_R - G_A)

    eigval_arr, eigvec_arr = LA.eigh(Spectr)
    
    return eigval_arr, eigvec_arr    

In [ ]:
def curr_matr_cont(x, i):
    """
    Constructs the current operator matrix for continuum quasiparticle
    states with energies |x| > Δ.

    The matrix is derived from the phase-dependent self-energy
    contributions of the left and right superconducting leads.
    For continuum states, the retarded branch of the square root
    √(x² − Δ²) is used in the denominator.

    Returns
    -------
    ndarray
        Current operator matrix evaluated at energy x.
    """

    s_L = 1
    s_R = -1
    zero_m = np.zeros_like(Gamma_L)
    
    Lambda_L = x * np.block([[Gamma_L , zero_m], [zero_m, Gamma_L]]) \
             - Delta * np.block([[zero_m, np.exp(-1j * s_L * phi_0[i]/2) * np.conjugate(np.transpose(F_L))], \
                                [np.exp(1j * s_L * phi_0[i]/2) * F_L, zero_m]]) 
    
    Lambda_R = x * np.block([[Gamma_R , zero_m], [zero_m, Gamma_R]]) \
         - Delta * np.block([[zero_m, np.exp(-1j * s_R * phi_0[i]/2) * np.conjugate(np.transpose(F_R))], \
                                [np.exp(1j * s_R * phi_0[i]/2) * F_R, zero_m]])
    
    I = -1 / (-1j * np.sign(x) * np.sqrt(x**2 - Delta**2)) / (2*1j) \
        * (s_L * Lambda_L + s_R * Lambda_R)
    
    return I

In [ ]:
def curr_cont_nl(E, E_lamd, psi_n, eta_lamd, i):
    """
    Computes the current matrix element between a continuum state and
    an Andreev bound state.

    The matrix element is evaluated between the continuum eigenvector ψ_n
    with energy E and the Andreev state eigenvector η_λ with energy E_λ.
    It involves the current operator matrices for continuum and bound
    states together with the τ_z matrix accounting for the particle–hole
    structure in Nambu space.

    Returns
    -------
    complex
        Current matrix element between the continuum state and the
        Andreev bound state.
    """
        
    matr = tau_z @ curr_matr(E_lamd, i) - curr_matr_cont(E, i) @ tau_z
    
    return np.conj(psi_n) @ matr @ eta_lamd    

In [ ]:
def integr_gamma_out(x, E_lamd, eta_lamd, i, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c):
    """
    Integrand for the ABS → continuum transition rate (Γ_out).

    For a given energy transfer x, the function evaluates transitions
    from an Andreev bound state with energy E_λ to continuum states
    with energy E = x + E_λ. The contribution is obtained by summing
    over all continuum eigenstates, weighted by their spectral density
    and the squared current matrix elements.

    The transition probability is multiplied by the bosonic bath
    occupation, the fermionic occupation factor of the continuum
    state, and the spectral density of the environment.

    Returns
    -------
    complex
        Value of the integrand entering the energy integral for the
        ABS → continuum transition rate.
    """
    
    E = x + E_lamd
    
    # 8 eigenvalues, eigenvectors for continuum state
    eigv_E, eigvect_E = spectr(E, i, MF_param)
    
    # current elements
    I_nl = np.zeros((8), dtype='complex')
    
    for n in range(8): 
        I_nl = curr_cont_nl(E, E_lamd, eigvect_E[:, n], eta_lamd, i)
        
    sum_n = np.sum(eigv_E * np.abs(I_nl)**2)
    
    n_F = fermi_distrib(E, T_ferm)
    n_B = bose_distrib(x, T_bos)
    J_sp = spectral_density(x, lambda_coupl, alpha_0, damping_param, Omega, omega_c)
    
    return J_sp * n_B * (1 - n_F) * sum_n 
    

In [ ]:
def integr_gamma_in(x, E_lamd, eta_lamd, i, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c):
    """
    Integrand for the continuum → ABS transition rate (Γ_in).

    For a given energy transfer x, the function evaluates transitions
    from continuum states with energy E = x + E_λ into an Andreev
    bound state with energy E_λ. The contribution is obtained by
    summing over all continuum eigenstates, weighted by their
    spectral density and the squared current matrix elements.

    The transition probability is multiplied by the bosonic emission
    factor (n_B + 1), the fermionic occupation of the continuum
    state, and the spectral density of the environment.

    Returns
    -------
    complex
        Value of the integrand entering the energy integral for the
        continuum → ABS transition rate.
    """
   
    E = x + E_lamd
    
    # 8 eigenvalues, eigenvectors for continuum state
    eigv_E, eigvect_E = spectr(E, i, MF_param)
    
    # current elements
    I_nl = np.zeros((8), dtype='complex')
    
    for n in range(8): 
        I_nl = curr_cont_nl(E, E_lamd, eigvect_E[:, n], eta_lamd, i)
        
    sum_n = np.sum(eigv_E * np.abs(I_nl)**2)
    
    n_F = fermi_distrib(E, T_ferm)
    n_B = bose_distrib(x, T_bos)
    J_sp = spectral_density(x, lambda_coupl, alpha_0, damping_param, Omega, omega_c)
    
    return J_sp * (n_B + 1) * n_F * sum_n 
    

In [ ]:
def compute_integrals(idx, i, lamd, Delta, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c):
    """
    Computes the energy integrals entering the transition rates between
    an Andreev bound state and continuum quasiparticle states.

    The function evaluates four contributions:
    - Γ_out_pc : ABS → positive-energy continuum
    - Γ_out_mc : ABS → negative-energy continuum
    - Γ_in_pc  : positive-energy continuum → ABS
    - Γ_in_mc  : negative-energy continuum → ABS

    The rates are obtained by integrating the corresponding transition
    rate integrands over the continuum energy ranges |E| > Δ.

    Returns
    -------
    tuple
        (idx, lamd, Gamma_out_pc, Gamma_out_mc, Gamma_in_pc, Gamma_in_mc)
        containing the computed transition rates for the given ABS index.
    """
    
    Gamma_out_pc = integrate.quad(integr_gamma_out, Delta - roots[i, lamd], np.infty,
                                  args=(roots[i, lamd], eta_arr[lamd, idx], i, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c))[0]
    
    Gamma_out_mc = integrate.quad(integr_gamma_out, -np.infty, -Delta - roots[i, lamd],
                                  args=(roots[i, lamd], eta_arr[lamd, idx], i, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c))[0]
    
    Gamma_in_pc = integrate.quad(integr_gamma_in, Delta - roots[i, lamd], np.infty,
                                 args=(roots[i, lamd], eta_arr[lamd, idx], i, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c))[0]
    
    Gamma_in_mc = integrate.quad(integr_gamma_in, -np.infty, -Delta - roots[i, lamd],
                                 args=(roots[i, lamd], eta_arr[lamd, idx], i, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c))[0]
    
    # Return results as a tuple
    return (idx, lamd, Gamma_out_pc, Gamma_out_mc, Gamma_in_pc, Gamma_in_mc)

In [ ]:
## Arrays for transition rates between ABSs and continuum:
##     - Γ_out_pc : ABS → positive-energy continuum
##     - Γ_out_mc : ABS → negative-energy continuum
##     - Γ_in_pc  : positive-energy continuum → ABS
##     - Γ_in_mc  : negative-energy continuum → ABS

## Note:
## For large x/T the exponential in the Bose distribution may overflow
## numerically (exp(x/T) → inf). In this regime n_B → 0, which is the
## physically correct limit. Therefore this warning does not affect the
## results and only corresponds to exponentially suppressed transitions.

Gamma_out_pc = np.zeros((len(phi_0), 4))
Gamma_out_mc = np.zeros_like(Gamma_out_pc) 
Gamma_in_pc = np.zeros_like(Gamma_out_pc)
Gamma_in_mc = np.zeros_like(Gamma_out_pc)

# Use ProcessPoolExecutor for parallel execution
with ProcessPoolExecutor() as executor:
    
    # Create a list of futures by submitting tasks for parallel execution
    futures = [executor.submit(compute_integrals, idx, i, lamd, Delta, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c)
               for idx, i in enumerate(range(len(phi_0))) for lamd in range(4)]
    
    # Collect results as they are completed
    for future in futures:
        i, lamd, out_pc, out_mc, in_pc, in_mc = future.result()
        
        # Store results in the appropriate arrays
        Gamma_out_pc[i, lamd] = out_pc
        Gamma_out_mc[i, lamd] = out_mc
        Gamma_in_pc[i, lamd] = in_pc
        Gamma_in_mc[i, lamd] = in_mc

In [ ]:
# Defining Γ_in and Γ_out transition rates as a sum over positive nd negative continuum.

Gamma_in = (Gamma_in_pc + Gamma_in_mc) * 2*np.pi
Gamma_out = (Gamma_out_pc + Gamma_out_mc) * 2*np.pi

In [ ]:
# Plotting the Γ_in and Γ_out transition rates

for i in range(Gamma_out.shape[1]):
    plt.plot(phi_0/np.pi, Gamma_out[:, i], label=rf'$\Gamma^{{out}}_{{{i+1}}}$')
    plt.plot(phi_0/np.pi, Gamma_in[:, i], label=rf'$\Gamma^{{in}}_{{{i+1}}}$')

plt.legend()
plt.grid()

plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)

## Lindblad equation: steady state

In this section we compute the **steady-state occupation probabilities** of the Andreev bound states using a **Lindblad master equation**.

The system contains **N = 4 fermionic levels**, giving a Hilbert space of  
\(2^N = 16\) many-body states corresponding to all possible occupation configurations.

Each many-body state is represented in **binary form**, where:
- `0` → level unoccupied  
- `1` → level occupied  

Using the transition rates previously computed (`Γ_in`, `Γ_out`, and Andreev–continuum processes), we construct the **rate matrix** of the master equation:

$\dot{\rho} = M \rho$

The **steady state** corresponds to the solution

$M \rho_{ss} = 0$

which is obtained by computing the **null space of the rate matrix** for each superconducting phase \( \phi_0 \).  
The resulting vector gives the **steady-state occupation probabilities of all 16 many-body states**.

In [ ]:
# number of fermionic levels and many-body states
N = 4

# number of many-body states
N_state = 2**N

In [ ]:
# returns the binary representation of a many-body state
# (occupation configuration of the N levels)

def print_state(n):
    
    if len(bin(n)[:1:-1]) < N:
        new_bin = bin(n)[:1:-1]
        for i in range(N - len(bin(n)[:1:-1])):
            new_bin += '0'
            
        return new_bin
    else:
        return bin(n)[:1:-1]

In [ ]:
# occupation function: returns whether level q is occupied
# in many-body state i

def n(q, i):
    if q > len(bin(i)[:1:-1])-1:
        return 0
    else:
        return int(bin(i)[:1:-1][q])

In [ ]:
lvl = np.array(range(N))
states = np.array(range(N_state))

n_v = np.vectorize(n)

# nf_h[state, level] → occupation (0 or 1) of level in that state
nf_h = np.array([n_v(lvl, si) for si in states])

In [ ]:
# rate matrix of the Lindblad master equation

# steady state matrix
Steady_M = np.zeros((len(phi_0), N_state, N_state), dtype='float64') 

for j in range(len(phi_0)):
    
    # loop over the raws - as many as there are states - 16
    for i in range(N_state):

        # loop over 4 levels
        for mu in range(4):

            n_mu = nf_h[i, mu]

            # G_in
            Steady_M[j, i, i + (-1)**n_mu * 2**mu] += n_mu * (Gamma_in[j, mu] )

            # G_out
            Steady_M[j, i, i + (-1)**n_mu * 2**mu] += (1-n_mu) * (Gamma_out[j, mu] )

            # G_in, G_out
            Steady_M[j, i, i] -= n_mu * (Gamma_out[j, mu]) + \
                                (1-n_mu) * (Gamma_in[j, mu] )

    #             loop over lambda' != lamda
            for nu in chain(range(mu), range(mu+1, 4)):

                n_nu = nf_h[i, nu]

                Steady_M[j, i, i + (-1)**n_mu * 2**mu + (-1)**n_nu * 2**nu] += \
                                n_mu * (1-n_nu) * G_AL_pp[j, nu, mu] + \
                                n_mu * n_nu * G_AL_mp[j, nu, mu] + \
                                (1-n_mu) * (1-n_nu) * G_AL_pm[j, nu, mu]

                Steady_M[j, i, i] -= (1-n_mu) * n_nu * G_AL_pp[j, nu, mu] + \
                                         (1-n_mu) * (1-n_nu) * G_AL_mp[j, nu, mu] +\
                                         n_mu * n_nu * G_AL_pm[j, nu, mu]


In [ ]:
# steady-state solution obtained from the null space of the rate matrix
ns = np.zeros((len(phi_0), 16), dtype='float64')

for i in range(len(phi_0)):
    ns[i] = null_space(Steady_M[i], rcond=10e-16)[:,0]
#     print(i, LA.matrix_rank(Steady_M[i]))
      
steady_occup = ns
steady_occup /= np.sum(steady_occup, axis=1, keepdims=True)

In [ ]:
for i in range(N_state):
    print(nf_h[i], '\t', steady_occup[25, i])

In [ ]:
# Plotting the steady state occupation probabilities

for i in range(12):
    plt.plot(phi_0/np.pi, steady_occup[:, i], label=rf'$P_{i}$')

plt.grid()
plt.legend()
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)

### Manual update of the mean-field parameters (diagnostic)

This section performs a **single manual update of the mean-field (MF) parameters**
using the steady-state occupations obtained from the Lindblad equation.

It was used to **diagnose and verify the convergence of the MFA loop** by:
- computing the occupation of each Andreev level,
- constructing the updated MF parameters,
- comparing them with the previous iteration.

*Note:* This block was used to check the mean-field convergence manually
before implementing the automated MFA loop.

In [ ]:
# compute weights of many-body states in terms of Andreev occupations
weight = np.einsum('km,ijm->ijk', nf_h, a_i)

# expectation value of the Andreev occupations
n_lambd = np.sum(weight * steady_occup[:, None, :], axis=2) + a

# construct MF update matrix from occupation differences
X_matr = n_lambd[:, :, None] - n_lambd[:, None, :]

# updated MF parameters
MF_param_new = np.sum(X_matr, axis=1)

# check convergence of MF parameters
abs_diff = np.abs(MF_param_new - MF_param)

# update the MF parameters
MF_param = MF_param_new

## Self-consistent Mean-Field Algorithm (MFA)

This section implements the **self-consistent mean-field (MF) loop** used to determine the interaction parameters of the quantum dot.

For each superconducting phase $ \varphi_0 $, the algorithm iteratively performs the following steps:

1. **Solve for Andreev bound state energies** by finding the zeros of $ \det(G^{-1}(x)) $.

2. **Diagonalize the effective Hamiltonian** to obtain eigenvalues and eigenvectors corresponding to the Andreev states.

3. **Construct the Bogoliubov transformation** that relates dot fermion operators to Andreev quasiparticles.

4. **Compute transition rates**
   - ABS → ABS transitions
   - ABS ↔ continuum transitions

5. **Build the Lindblad rate matrix** describing transitions between the many-body states of the system.

6. **Solve the steady-state master equation** to obtain the occupation probabilities of the many-body states.

7. **Update the mean-field parameters** from the expectation values of the Andreev occupations.

The procedure is repeated until the MF parameters **converge within a given tolerance**.

### Efficiency improvement

Since convergence occurs at different speeds for different values of $ \varphi_0 $, the algorithm tracks which phase points have already converged.  
Only the **non-converged indices are updated in subsequent iterations**, significantly reducing the computational cost of the loop.

In [ ]:
# initializing the MFA procedure

MF_param = np.zeros((len(phi_0), 2*l), dtype='complex')
abs_diff = np.full((len(phi_0), 2*l), np.inf)

accur_par = 6e-3
max_iter = 10
MFA_count = 0

active_mask = np.full(len(phi_0), True)

In [ ]:
# Self-consistent MFA loop with adaptive reduction of non-converged phase indices

while MFA_count < max_iter and np.any(active_mask):

    # Select indices where not converged
    active_indices = np.where(active_mask)[0]
    
    # You may want to skip empty active_indices check for safety
    if len(active_indices) == 0:
        break
    
    # --- 1. Compute all roots, will use only active further ---
    roots = np.zeros((len(phi_0), 4*l))   
    
    for i in range(len(phi_0)):
        roots[i, 0] = fsolve(G_inv, 0.55, args=(i, MF_param))
        roots[i, 1] = fsolve(G_inv, 0.7, args=(i, MF_param))
        roots[i, 2] = fsolve(G_inv, 0.8, args=(i, MF_param))
        roots[i, 3] = fsolve(G_inv, 0.99, args=(i, MF_param))
        roots[i, 4] = fsolve(G_inv, -0.55, args=(i, MF_param))
        roots[i, 5] = fsolve(G_inv, -0.7, args=(i, MF_param))
        roots[i, 6] = fsolve(G_inv, -0.8, args=(i, MF_param))
        roots[i, 7] = fsolve(G_inv, -0.99, args=(i, MF_param))

    # --- 2. Compute eigenvalues/eigenvectors for active indices ---
    
    eigv_ = np.zeros((len(active_indices), 4*l, 4*l))
    eigvec_ = np.zeros((len(active_indices), 4*l, 4*l, 4*l), dtype='complex')

    for k in range(4*l):
        for idx, i in enumerate(active_indices):
            eigv_[idx, :, k], eigvec_[idx, :, :, k] = diagonal(roots[:, k], i, MF_param)

    
    # Eigenvectors corresponding to the Andreev energies
    eta_0 = eigvec_[:, :, 4, 0]
    eta_1 = eigvec_[:, :, 5, 1]
    eta_2 = eigvec_[:, :, 6, 2]
    eta_3 = eigvec_[:, :, 7, 3]

    eta_4 = eigvec_[:, :, 3, 4]
    eta_5 = eigvec_[:, :, 2, 5]
    eta_6 = eigvec_[:, :, 1, 6]
    eta_7 = eigvec_[:, :, 0, 7]

    eta_arr = np.stack([eta_0, eta_1, eta_2, eta_3, eta_4, eta_5, eta_6, eta_7], axis=0)

    # --- 3. Compute eta_arr, M, M_inv, a_i, etc. for active_indices ---
    # Make sure these computations only use active_indices slices
    
    # Bogoliubov transformation for the dot -> Andreev fermions 
    M = np.zeros((len(active_indices), 8, 8), dtype='complex')

    # Fill M[:, 0:4, :] with the conjugates of eta_0 to eta_3
    M[:, 0, :] = np.conj(eta_0)
    M[:, 1, :] = np.conj(eta_1)
    M[:, 2, :] = np.conj(eta_2)
    M[:, 3, :] = np.conj(eta_3)

    # For rows 4 to 7, split and assign parts
    for i, eta in enumerate([eta_0, eta_1, eta_2, eta_3]):
        M[:, 4 + i, 0:4] = eta[:, 4:]
        M[:, 4 + i, 4:] = eta[:, 0:4]

    M_inv = LA.inv(M)
    a = np.sum(np.abs(M_inv[:, 0:4, 4:])**2, axis=1)

    # weight of each number of part operator n_i, i in [1, 4]
    a_i = np.abs(M_inv[:, 0:4, 0:4])**2 - np.abs(M_inv[:, 4:, 0:4])**2

    # --- 4. Compute current elements I_nl and rates G_nl for active_indices ---
    
    # Current elements
    I_nl = np.zeros((len(active_indices), 4*l, 4*l), dtype='complex')
    G_nl = np.zeros((len(active_indices), 4*l, 4*l))

    for idx, i in enumerate(active_indices):
        for n in range(4*l):
            for lamd in range(4*l):
                I_nl[idx, n, lamd] = curr_nl(
                    roots[i, n],
                    roots[i, lamd],
                    eta_arr[n, idx],
                    eta_arr[lamd, idx],
                    i)
                
    # ABS -> ABS transition rates
    for idx, i in enumerate(active_indices):
        for n in range(4*l):
            for lamd in range(4*l):

                energy_diff = roots[i, n] - roots[i, lamd]
                J_spect = spectral_density(energy_diff, lambda_coupl, alpha_0, damping_param, Omega, omega_c)
                bose = bose_distrib(energy_diff, T_bos)

                G_nl[idx, lamd, n] = 2 * np.pi * np.abs(I_nl[idx, n, lamd])**2 * J_spect * bose

                if math.isnan(G_nl[idx, lamd, n]):
                    G_nl[idx, lamd, n] = 0
    
    
    G_AL_pp = G_nl[:, 0:4, 0:4]
    G_AL_pm = G_nl[:, 0:4, 4:8]
    G_AL_mp = G_nl[:, 4:8, 0:4]

    
    # --- 5. Compute Gamma_in_pc, Gamma_out_pc, Gamma_in_mc, Gamma_out_mc ---
    
    # ABS -> continuum transition rates 
    Gamma_out_pc = np.zeros((len(active_indices), 4))
    Gamma_out_mc = np.zeros_like(Gamma_out_pc) 
    Gamma_in_pc = np.zeros_like(Gamma_out_pc)
    Gamma_in_mc = np.zeros_like(Gamma_out_pc)

    # Use ProcessPoolExecutor for parallel execution
    with ProcessPoolExecutor() as executor:

        # Create a list of futures by submitting tasks for parallel execution
        futures = [executor.submit(compute_integrals, idx, i, lamd, Delta, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c)
               for idx, i in enumerate(active_indices) for lamd in range(4)]
            
        # Collect results as they are completed
        for future in futures:
            idx, lamd, out_pc, out_mc, in_pc, in_mc = future.result()

            Gamma_out_pc[idx, lamd] = out_pc
            Gamma_out_mc[idx, lamd] = out_mc
            Gamma_in_pc[idx, lamd]  = in_pc
            Gamma_in_mc[idx, lamd]  = in_mc

            
    Gamma_in = (Gamma_in_pc + Gamma_in_mc) * 2*np.pi
    Gamma_out = (Gamma_out_pc + Gamma_out_mc) * 2*np.pi
    
    # --- 6. Build Steady_M matrices for active_indices ---
    
    # Steady state 
    Steady_M = np.zeros((len(active_indices), N_state, N_state), dtype='float64') 

    for idx, j in enumerate(active_indices):
        # loop over the raws - as many as there are states - 16
        for i in range(N_state):

            # loop over 4 levels
            for mu in range(4):

                n_mu = nf_h[i, mu]

                # G_in
                Steady_M[idx, i, i + (-1)**n_mu * 2**mu] += n_mu * (Gamma_in[idx, mu] )

                # G_out
                Steady_M[idx, i, i + (-1)**n_mu * 2**mu] += (1-n_mu) * (Gamma_out[idx, mu] )

                # G_in, G_out
                Steady_M[idx, i, i] -= n_mu * (Gamma_out[idx, mu]) + \
                                    (1-n_mu) * (Gamma_in[idx, mu] )

        #             loop over lambda' != lamda
                for nu in chain(range(mu), range(mu+1, 4)):

                    n_nu = nf_h[i, nu]

                    Steady_M[idx, i, i + (-1)**n_mu * 2**mu + (-1)**n_nu * 2**nu] += \
                                    n_mu * (1-n_nu) * G_AL_pp[idx, nu, mu] + \
                                    n_mu * n_nu * G_AL_mp[idx, nu, mu] + \
                                    (1-n_mu) * (1-n_nu) * G_AL_pm[idx, nu, mu]

                    Steady_M[idx, i, i] -= (1-n_mu) * n_nu * G_AL_pp[idx, nu, mu] + \
                                             (1-n_mu) * (1-n_nu) * G_AL_mp[idx, nu, mu] +\
                                             n_mu * n_nu * G_AL_pm[idx, nu, mu]
    
    # --- 7. Compute steady state occupation ns for active_indices ---
    
    ns = np.zeros((len(active_indices), 16), dtype='float64')

    for idx, i in enumerate(active_indices):
        ns[idx] = null_space(Steady_M[idx], rcond=1e-16)[:, 0]
        
    steady_occup = ns
    steady_occup /= np.sum(steady_occup, axis=1, keepdims=True)
    
    # --- 8. Compute new MF_param for active_indices ---
   
    # Computing accuracy and updating MF_param 
    weight = np.einsum('km,ijm->ijk', nf_h, a_i)
    n_lambd = np.sum(weight * steady_occup[:, None, :], axis=2) + a
    
    X_matr = n_lambd[:, :, None] - n_lambd[:, None, :]
    MF_param_new_active = np.sum(X_matr, axis=1)

    # --- 9. Update abs_diff for active_indices ---
    abs_diff[active_indices] = np.abs(MF_param_new_active - MF_param[active_indices])

    # --- 10. Update MF_param for active_indices ---
    MF_param[active_indices] = MF_param_new_active

    # --- 11. Update active_mask: mark converged points as False ---
    converged = np.all(abs_diff[active_indices] <= accur_par, axis=1)
    for idx, conv in zip(active_indices, converged):
        if conv:
            active_mask[idx] = False

    MFA_count += 1
    
    print(f"Iteration {MFA_count}, points left to converge: {np.sum(active_mask)}")
    print("Max difference among active points:", np.max(abs_diff[active_indices]))


### Numerical note

During the MFA iterations some intermediate quantities (in particular the
Bose distribution appearing in transition rates) may produce numerical
overflow warnings for large energy differences (exp(x/T) → ∞).

In this regime the Bose factor tends to zero and the corresponding
transition rates become exponentially suppressed. Therefore these
warnings do not affect the physical results and can be safely ignored.

### Convergency note

During testing of the self-consistent MFA loop, the mean-field parameters
did not converge for all parameter regimes explored in this notebook.
The divergence appears to be related to the stability of the
self-consistency procedure and requires further investigation.

Since the project was not completed, this issue remains unresolved.
The code is nevertheless included to demonstrate the full numerical
pipeline used to compute the Andreev spectrum, transition rates,
and steady-state occupations.

In [ ]:
# plotting the resulting MF parameter

plt.plot(phi_0/np.pi, MF_param)

plt.grid()
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)
plt.ylabel(r'$MF parameters$', fontsize=12)

In [ ]:
# plotting the resulting absolute difference

plt.plot(phi_0/np.pi, abs_diff)
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)

In [ ]:
# saving/loading the final MF parameters

MF_param_final = MF_param

# np.save("MF_param_bx=0.21,bz=0.2, SO=0.1.npy", MF_param_final)
# MF_param_final = np.load("MF_param_bx=0.21,bz=0.2, SO=0.1.npy") # loading MF_param

## Current calculation

After obtaining the steady-state occupation probabilities of the Andreev states, the Josephson current is evaluated. The total current consists of two contributions:

- **Andreev bound state current** arising from the phase dependence of the discrete Andreev spectrum.
- **Continuum current** originating from quasiparticle states above the superconducting gap.

The Andreev contribution is computed from the phase derivative of the Andreev energies

$ I_A(\varphi) = \sum_n \frac{\partial E_n}{\partial \varphi}\,(2 w_n - 1),$

where $E_n$ are the Andreev energies and $w_n$ are the steady-state occupation probabilities obtained from the master equation.

The continuum contribution is calculated by integrating over the quasiparticle spectrum above the superconducting gap.

The total current is therefore

$I(\varphi) = I_A(\varphi) + I_{\text{cont}}(\varphi).$

Finally, the **superconducting diode efficiency**

$\eta = \frac{|I_{+} - I_{-}|}{I_{+} + I_{-}}$

is evaluated to quantify the difference between the positive and negative critical currents.

In [ ]:
roots = np.zeros((len(phi_0), 4*l))

for i in range(len(phi_0)):
    
    roots[i, 0] = fsolve(G_inv, 0.55, args=(i, MF_param_final))  
    roots[i, 1] = fsolve(G_inv, 0.7, args=(i, MF_param_final))  #0.65   
    roots[i, 2] = fsolve(G_inv, 0.8, args=(i, MF_param_final))  #0.86
    roots[i, 3] = fsolve(G_inv, 0.99, args=(i, MF_param_final))

    roots[i, 4] = fsolve(G_inv, -0.55, args=(i, MF_param_final))  
    roots[i, 5] = fsolve(G_inv, -0.7, args=(i, MF_param_final))     
    roots[i, 6] = fsolve(G_inv, -0.8, args=(i, MF_param_final))  
    roots[i, 7] = fsolve(G_inv, -0.99, args=(i, MF_param_final)) 

In [ ]:
plt.plot(phi_0/np.pi, roots[:,0], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,1], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,2], linewidth=1.5)
plt.plot(phi_0/np.pi, roots[:,3], linewidth=1.5)


plt.title(r"$t={}, E_C={}, \gamma_{{SO}} = {},$"\
          .format(t_L, Ec, gamma_SO)+'\n'+
          r"$b_x={}\Delta, b_z = {}\Delta, L={}\Delta^{{-1}}$"\
          .format(np.round(b_x, 2), np.round(b_z, 2), L), fontsize=13)



plt.grid()
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)
plt.ylabel(r'$E_A$, $\Delta$', fontsize=12)

In [ ]:
eigv_ = np.zeros((len(phi_0), 4*l, 4*l))
eigvec_ = np.zeros((len(phi_0), 4*l, 4*l, 4*l), dtype='complex')

for k in range(4*l):
    for i in range(len(phi_0)):
        eigv_[i, :, k], eigvec_[i, :, :, k] = diagonal(roots[:, k], i, MF_param_final)


# Eigenvectors corresponding to the Andreev energies
eta_0 = eigvec_[:, :, 4, 0]
eta_1 = eigvec_[:, :, 5, 1]
eta_2 = eigvec_[:, :, 6, 2]
eta_3 = eigvec_[:, :, 7, 3]

eta_4 = eigvec_[:, :, 3, 4]
eta_5 = eigvec_[:, :, 2, 5]
eta_6 = eigvec_[:, :, 1, 6]
eta_7 = eigvec_[:, :, 0, 7]

eta_arr = np.stack([eta_0, eta_1, eta_2, eta_3, eta_4, eta_5, eta_6, eta_7], axis=0)

In [ ]:
# Current elements
    
I_nl = np.zeros((len(phi_0), 4*l, 4*l), dtype='complex')
G_nl = np.zeros((len(phi_0), 4*l, 4*l))

for i in range(len(phi_0)):
    for n in range(4*l):
        for lamd in range(4*l):
            I_nl[i, n ,lamd] = curr_nl(roots[i, n], roots[i, lamd], eta_arr[n, i], eta_arr[lamd, i], i)

# ABS -> ABS transition rates

for i in range(len(phi_0)):
    for n in range(4*l):
        for lamd in range(4*l):

            energy_diff = roots[i, n] - roots[i, lamd]
            J_spect = spectral_density(energy_diff, lambda_coupl, alpha_0, damping_param, Omega, omega_c)
            bose = bose_distrib(energy_diff, T_bos)

            G_nl[i, lamd, n] = 2*np.pi * np.abs(I_nl[i, n, lamd])**2 * J_spect * bose 

            if math.isnan(G_nl[i, lamd, n]):
                G_nl[i, lamd, n] = 0


G_AL_pp = G_nl[:, 0:4, 0:4]
G_AL_pm = G_nl[:, 0:4, 4:8]
G_AL_mp = G_nl[:, 4:8, 0:4]

In [ ]:
# ABS -> continuum transition rates
    
Gamma_out_pc = np.zeros((len(phi_0), 4))
Gamma_out_mc = np.zeros_like(Gamma_out_pc) 
Gamma_in_pc = np.zeros_like(Gamma_out_pc)
Gamma_in_mc = np.zeros_like(Gamma_out_pc)

# Use ProcessPoolExecutor for parallel execution
with ProcessPoolExecutor() as executor:
    
    # Create a list of futures by submitting tasks for parallel execution
    futures = [executor.submit(compute_integrals, idx, i, lamd, Delta, T_bos, T_ferm,\
                    lambda_coupl, alpha_0, damping_param, Omega, omega_c)
               for idx, i in enumerate(range(len(phi_0))) for lamd in range(4)]
    
    # Collect results as they are completed
    for future in futures:
        i, lamd, out_pc, out_mc, in_pc, in_mc = future.result()
        
        # Store results in the appropriate arrays
        Gamma_out_pc[i, lamd] = out_pc
        Gamma_out_mc[i, lamd] = out_mc
        Gamma_in_pc[i, lamd] = in_pc
        Gamma_in_mc[i, lamd] = in_mc


Gamma_in = (Gamma_in_pc + Gamma_in_mc) * 2*np.pi
Gamma_out = (Gamma_out_pc + Gamma_out_mc) * 2*np.pi

plt.plot(phi_0/np.pi, Gamma_out[:, 0], label=r'$\Gamma^{out}_1$')
plt.plot(phi_0/np.pi, Gamma_in[:, 0], label=r'$\Gamma^{in}_1$')

plt.plot(phi_0/np.pi, Gamma_out[:, 1], label=r'$\Gamma^{out}_{2}$')
plt.plot(phi_0/np.pi, Gamma_in[:, 1], label=r'$\Gamma^{in}_{2}$')

plt.legend()
plt.grid()

# Note: During the evaluation of the Bose and Fermi distributions,
# overflow warnings may appear for large |E/T| values. These correspond
# to exponentially suppressed occupation factors and do not affect the
# final transition rates or current results.

In [ ]:
# Steady state
    
Steady_M = np.zeros((len(phi_0), N_state, N_state), dtype='float64') 

for j in range(len(phi_0)):

    # loop over the raws - as many as there are states - 16
    for i in range(N_state):

        # loop over 4 levels
        for mu in range(4):

            n_mu = nf_h[i, mu]

            # G_in
            Steady_M[j, i, i + (-1)**n_mu * 2**mu] += n_mu * (Gamma_in[j, mu] )

            # G_out
            Steady_M[j, i, i + (-1)**n_mu * 2**mu] += (1-n_mu) * (Gamma_out[j, mu] )

            # G_in, G_out
            Steady_M[j, i, i] -= n_mu * (Gamma_out[j, mu]) + \
                                (1-n_mu) * (Gamma_in[j, mu] )

    #             loop over lambda' != lamda
            for nu in chain(range(mu), range(mu+1, 4)):

                n_nu = nf_h[i, nu]

                Steady_M[j, i, i + (-1)**n_mu * 2**mu + (-1)**n_nu * 2**nu] += \
                                n_mu * (1-n_nu) * G_AL_pp[j, nu, mu] + \
                                n_mu * n_nu * G_AL_mp[j, nu, mu] + \
                                (1-n_mu) * (1-n_nu) * G_AL_pm[j, nu, mu]

                Steady_M[j, i, i] -= (1-n_mu) * n_nu * G_AL_pp[j, nu, mu] + \
                                         (1-n_mu) * (1-n_nu) * G_AL_mp[j, nu, mu] +\
                                         n_mu * n_nu * G_AL_pm[j, nu, mu]



In [ ]:
ns = np.zeros((len(phi_0), 16), dtype='float64')

for i in range(len(phi_0)):
    ns[i] = null_space(Steady_M[i], rcond=10e-16)[:,0]
#         print(i, LA.matrix_rank(Steady_M[i]))

steady_occup_final = ns
steady_occup_final /= np.sum(steady_occup_final, axis=1, keepdims=True)
    
for i in range(6):
    plt.plot(phi_0/np.pi, steady_occup_final[:, i], label=f'$P_{i}$')
    
plt.grid()
plt.legend()

In [ ]:
# save or load the steady state

np.save("steady_occup_final_bx,bz=0.1, SO=0.2.npy", steady_occup_final)
# steady_occup_final = np.load("steady_occup_final_bx,bz=0.1, SO=0.2.npy") # loading MF_param

In [ ]:
w_lamb = np.sum(nf_h[None, :, :] * steady_occup_final[:, :, None], axis=1)

In [ ]:
def current_cont(E, MF_param, dx, T_ferm):
    
    eigv_c = np.zeros((len(phi_0), 8))
    eigvect_c = np.zeros((len(phi_0), 8, 8), dtype='complex')

    for i in range(len(phi_0)):
        eigv_c[i], eigvect_c[i] = spectr(E, i, MF_param)
    
    grad_c = np.gradient(eigv_c, dx, axis=0, edge_order=2)
    sum_grad = np.sum(grad_c[:, 0:4], axis=1)
    
    return sum_grad * (2*fermi_distrib(E, T_ferm) - 1)

In [ ]:
dx = 2*np.pi/len(phi_0)
grad = np.gradient(roots, dx, axis=0, edge_order=2)

# Andreev current
I_A = np.sum(grad[:, 0:4] * (2*w_lamb - 1), axis=1) 

# continuum current
I_cont = quad_vec(current_cont, Delta, np.infty, args=(MF_param_final, dx, T_ferm))[0]

# total current
I = I_A + I_cont

In [ ]:
plt.plot(phi_0/np.pi, I, label='total = A+c')
plt.plot(phi_0/np.pi, I_A, label='Andr')
plt.plot(phi_0/np.pi, I_cont, label='cont')

plt.grid()
plt.legend()

In [ ]:
I_pl = np.max(I)
I_min = np.max(-I)

eta = np.abs(I_pl - I_min)/(I_pl + I_min)
print('eta = ', eta*100, "%")

In [ ]:
I_A_pl = np.max(I_A)
I_A_min = np.max(-I_A)

eta_A = np.abs(I_A_pl - I_A_min)/(I_A_pl + I_A_min)
print('eta_A = ', eta_A*100, "%")